<a href="https://colab.research.google.com/github/zyf-hitsz/PytorchLearning/blob/main/NOTEBOOKS/OptimizingModelParameters/optimization_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

Optimizing Model Parameters
===========================

Now that we have a model and data it\'s time to train, validate and test
our model by optimizing its parameters on our data. Training a model is
an iterative process; in each iteration the model makes a guess about
the output, calculates the error in its guess (*loss*), collects the
derivatives of the error with respect to its parameters (as we saw in
the [previous section](autogradqs_tutorial.html)), and **optimizes**
these parameters using gradient descent. For a more detailed walkthrough
of this process, check out this video on [backpropagation from
3Blue1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8).

Prerequisite Code
-----------------

We load the code from the previous sections on [Datasets &
DataLoaders](data_tutorial.html) and [Build
Model](buildmodel_tutorial.html).


In [10]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

#加载训练数据集
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

#加载测试数据集
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

#创建训练数据加载器。它将training_data分成大小为64的小批量（batch）数据，并在训练时提供给模型。
train_dataloader = DataLoader(training_data, batch_size=64)
#创建测试数据加载器，同样将test_data分成64大小的批次。
test_dataloader = DataLoader(test_data, batch_size=64)

#定义神经网络模型
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

#实例化模型
model = NeuralNetwork()

Hyperparameters
===============

Hyperparameters are adjustable parameters that let you control the model
optimization process. Different hyperparameter values can impact model
training and convergence rates ([read
more](https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html)
about hyperparameter tuning)

We define the following hyperparameters for training:

:   -   **Number of Epochs** - the number of times to iterate over the
        dataset
    -   **Batch Size** - the number of data samples propagated through
        the network before the parameters are updated
    -   **Learning Rate** - how much to update models parameters at each
        batch/epoch. Smaller values yield slow learning speed, while
        large values may result in unpredictable behavior during
        training.


In [11]:
learning_rate = 1e-3#学习率：决定了模型在每次迭代中更新其参数的幅度，这里是0.001
batch_size = 64#批量大小：在每次模型参数更新前，用于计算梯度的数据样本数量，这里表示每次训练迭代会处理64个数据样本，然后根据这64个样本的平均损失来更新模型参数。
epochs = 5#训练轮次：一个“epoch”指的是模型对整个训练数据集进行一次完整的遍历，这里学习5次，在每个epoch中，模型会处理训练数据集中的所有批次数据。

Optimization Loop
=================

Once we set our hyperparameters, we can then train and optimize our
model with an optimization loop. Each iteration of the optimization loop
is called an **epoch**.

Each epoch consists of two main parts:

:   -   **The Train Loop** - iterate over the training dataset and try
        to converge to optimal parameters.
    -   **The Validation/Test Loop** - iterate over the test dataset to
        check if model performance is improving.

Let\'s briefly familiarize ourselves with some of the concepts used in
the training loop. Jump ahead to see the
`full-impl-label`{.interpreted-text role="ref"} of the optimization
loop.

Loss Function
-------------

When presented with some training data, our untrained network is likely
not to give the correct answer. **Loss function** measures the degree of
dissimilarity of obtained result to the target value, and it is the loss
function that we want to minimize during training. To calculate the loss
we make a prediction using the inputs of our given data sample and
compare it against the true data label value.

Common loss functions include
[nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html#torch.nn.MSELoss)
(Mean Square Error) for regression tasks, and
[nn.NLLLoss](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html#torch.nn.NLLLoss)
(Negative Log Likelihood) for classification.
[nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss)
combines `nn.LogSoftmax` and `nn.NLLLoss`.

We pass our model\'s output logits to `nn.CrossEntropyLoss`, which will
normalize the logits and compute the prediction error.


In [12]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

Optimizer
=========

Optimization is the process of adjusting model parameters to reduce
model error in each training step. **Optimization algorithms** define
how this process is performed (in this example we use Stochastic
Gradient Descent). All optimization logic is encapsulated in the
`optimizer` object. Here, we use the SGD optimizer; additionally, there
are many [different
optimizers](https://pytorch.org/docs/stable/optim.html) available in
PyTorch such as ADAM and RMSProp, that work better for different kinds
of models and data.

We initialize the optimizer by registering the model\'s parameters that
need to be trained, and passing in the learning rate hyperparameter.


In [13]:
#这里采用随机梯度下降（SGD）的优化算法。
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
#model 是我们之前定义的 NeuralNetwork 实例，model.parameters() 方法会返回一个迭代器，其中包含了模型中所有需要学习的参数
#lr 是 learning rate的缩写，这里将设定好的学习率传递给优化器

Inside the training loop, optimization happens in three steps:

:   -   Call `optimizer.zero_grad()` to reset the gradients of model
        parameters. Gradients by default add up; to prevent
        double-counting, we explicitly zero them at each iteration.
    -   Backpropagate the prediction loss with a call to
        `loss.backward()`. PyTorch deposits the gradients of the loss
        w.r.t. each parameter.
    -   Once we have our gradients, we call `optimizer.step()` to adjust
        the parameters by the gradients collected in the backward pass.


Full Implementation {#full-impl-label}
===================

We define `train_loop` that loops over our optimization code, and
`test_loop` that evaluates the model\'s performance against our test
data.


In [14]:
#训练循环
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)#获取训练数据集的总样本数。

    model.train()#设置为训练模式
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)#前向计算
        loss = loss_fn(pred, y)#损失值

        # Backpropagation
        loss.backward()#反向传播
        optimizer.step()#优化器步进，更新参数
        optimizer.zero_grad()#梯度清零

        #每处理100个批次打印一次损失值和处理进度
        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


#测试循环
def test_loop(dataloader, model, loss_fn):
    model.eval()#设置为评估模式
    size = len(dataloader.dataset)#获取测试集的总样本数
    num_batches = len(dataloader)#获取测试数据批次的数量
    test_loss, correct = 0, 0#初始化测试loss和正确预测的数量


    with torch.no_grad():#禁用梯度计算
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

We initialize the loss function and optimizer, and pass it to
`train_loop` and `test_loop`. Feel free to increase the number of epochs
to track the model\'s improving performance.


In [15]:
loss_fn = nn.CrossEntropyLoss()#初始化损失函数为交叉熵损失函数
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)#初始化优化器

epochs = 20
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.298964  [   64/60000]
loss: 2.296213  [ 6464/60000]
loss: 2.275636  [12864/60000]
loss: 2.280810  [19264/60000]
loss: 2.261254  [25664/60000]
loss: 2.224793  [32064/60000]
loss: 2.237422  [38464/60000]
loss: 2.199362  [44864/60000]
loss: 2.202383  [51264/60000]
loss: 2.176392  [57664/60000]
Test Error: 
 Accuracy: 43.7%, Avg loss: 2.169408 

Epoch 2
-------------------------------
loss: 2.172850  [   64/60000]
loss: 2.171275  [ 6464/60000]
loss: 2.112572  [12864/60000]
loss: 2.138319  [19264/60000]
loss: 2.086899  [25664/60000]
loss: 2.023684  [32064/60000]
loss: 2.053029  [38464/60000]
loss: 1.974717  [44864/60000]
loss: 1.982739  [51264/60000]
loss: 1.917048  [57664/60000]
Test Error: 
 Accuracy: 59.8%, Avg loss: 1.912931 

Epoch 3
-------------------------------
loss: 1.939896  [   64/60000]
loss: 1.916250  [ 6464/60000]
loss: 1.799580  [12864/60000]
loss: 1.848098  [19264/60000]
loss: 1.741206  [25664/60000]
loss: 1.686080  [32064/600

Further Reading
===============

-   [Loss
    Functions](https://pytorch.org/docs/stable/nn.html#loss-functions)
-   [torch.optim](https://pytorch.org/docs/stable/optim.html)
-   [Warmstart Training a
    Model](https://pytorch.org/tutorials/recipes/recipes/warmstarting_model_using_parameters_from_a_different_model.html)
